# Realizando transformações na bronze e salvando na silver 🥈
- O objetivo dessa camada é analisar os dados presentes na `bronze` e realizar operações encima deles, otimizando e tratando-os a fim de colocar dentro da camada `silver`
- É aqui que são feitos os tratamentos dos dados a fim de levar algo mais concreto pra dentro da `gold`

In [0]:
# Imports realizados pra tratamento das colunas e uso de Window Functions 
from pyspark.sql import functions as F 
from pyspark.sql.window import Window

In [0]:
# Definição das variáveis de ambiente que serão usadas ao longo dos arquivos 
catalogo = "medalhao_at"
schema_bronze = "bronze"
schema_silver = "silver"

# A função abaixo é uma função simples apenas pra ler as tabelas e receber um dataframe spark
def devolve_tabela_spark(nome_tabela):
    return spark.table(f"{catalogo}.{schema_bronze}.{nome_tabela}")

## 1️⃣ silver.dim_consumidores

In [0]:
# Leitura dos dados e análise de suas informações (Dados propriramente ditos e seu schema)
bronze_tb_customers = devolve_tabela_spark("tb_customers")
bronze_tb_customers.display(10)
bronze_tb_customers.printSchema()

# Também foi feita uma análise da existência de nulos e duplicatas completas dentro da tabela 
nulos_tb_customers = bronze_tb_customers.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_customers.columns
]).display(10)

# Verificação de linhas duplicadas baseado na coluna especificada 
total_linhas = bronze_tb_customers.count()
total_ids_unicos = bronze_tb_customers.select("customer_unique_id").distinct().count()

print(f"Total de linhas na Bronze: {total_linhas}")
print(f"Total de IDs únicos: {total_ids_unicos}")

In [0]:
# OBS: Foi optado pelo uso da coluna id_unico_consumidor devido ao fato de que, além de seu nome indicar que é uma coluna onde devem haver apenas id's únicos, a coluna id_consumidor a coluna id_consumidor não havia duplicatas, apenas na id_unico_consumidor

# Window Function que utilizada para filtrar a bronze procurando por duplicatas na coluna id_consumidor usando como referência pra odernação a coluna de data_ingestao 
window_spec = Window.partitionBy("id_unico_consumidor").orderBy(F.col("data_ingestao").desc())

# -----------------------
# Limpeza e padronização inicial 
# -----------------------
tb_customers_limpo = (
    bronze_tb_customers
    .select(
        F.col("customer_id").cast("string").alias("id_consumidor"),
        F.col("customer_unique_id").cast("string").alias("id_unico_consumidor"),
        F.col("customer_zip_code_prefix").cast("integer").alias("prefixo_cep"),
        F.upper(F.col("customer_city").cast("string")).alias("cidade"),
        F.upper(F.col("customer_state").cast("string")).alias("estado"),
        F.initcap(F.trim(F.col("customer_name"))).cast("string").alias("nome_consumidor"),
        F.upper(F.col("customer_gender").cast("string")).alias("genero"),
        F.to_date("customer_birth_date").alias("data_nascimento"),
        F.col("customer_age").cast("integer").alias("idade"),
        F.to_timestamp("timestamp_ingestion").alias("data_ingestao")
    )
    # 1. Aplicando a window function e armazenando em coluna temporária 
    .withColumn("rn", F.row_number().over(window_spec))
    
    # 2. Filtramos mantendo APENAS a linha número 1 (a mais recente de cada ID)
    .filter(F.col("rn") == 1)
    
    # 3. A coluna 'rn' foi removida pois ela era apenas um apoio para nos ajudar no filtro
    .drop("rn")
)

display(tb_customers_limpo.limit(10))

print(f"Total de linhas na silver (tb_customers): {tb_customers_limpo.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
tb_customers_validacao = (
    tb_customers_limpo
    .withColumn("flag_id_valido", F.col("id_consumidor").isNotNull())
    .withColumn("flag_id_unico_valido", F.col("id_unico_consumidor").isNotNull())
    .withColumn("flag_cep_valido", F.col("prefixo_cep") > 0)
    .withColumn("flag_nome_valido", F.col("nome_consumidor").isNotNull() & (F.length(F.col("nome_consumidor")) > 1))
    .withColumn("flag_idade_valida", F.col("idade").between(1, 119))
    .withColumn("flag_genero_valido", F.col("genero").isin("M", "F"))
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_id_valido") &
            F.col("flag_id_unico_valido") &
            F.col("flag_cep_valido") &
            F.col("flag_nome_valido") &
            F.col("flag_idade_valida") &
            F.col("flag_genero_valido"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

display(tb_customers_validacao.limit(10))

# -----------------------
# Separação dos dados em validos e invalidos
# -----------------------
tb_customers_validos = tb_customers_validacao.filter(F.col("flag_qualidade") == "OK")
tb_customers_invalidos = tb_customers_validacao.filter(F.col("flag_qualidade") == "ERRO")

# -----------------------
# Escrita na camada silver
# -----------------------

# Dados válidos
tb_customers_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_consumidores_validos")

# Dados inválidos (mantidos para auditoria e rastreabilidade)
tb_customers_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_consumidores_invalidos")

print(f"✅ Tabela silver.dim_consumidores_validos criada com sucesso! ({tb_customers_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.dim_consumidores criada para auditoria ({tb_customers_invalidos.count()} registros inválidos)")

## 2️⃣ silver.fat_pedidos

In [0]:
bronze_tb_orders = devolve_tabela_spark("tb_orders")
bronze_tb_orders.display(10)
bronze_tb_orders.printSchema()

nulos_tb_orders = bronze_tb_orders.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_orders.columns
]).display(10)

# Agurpando os nulos baseados na coluna do status do pedido pra entendermos o funcionamento desses dados
print("Quantidade de linhas nulos na coluna order_delivered_customer_date ordenando por status")
df = bronze_tb_orders
df_apenas_nulos = df.filter(F.col("order_delivered_customer_date").isNull())
display(df_apenas_nulos.groupBy("order_status").count())

In [0]:
# -----------------------
# Limpeza e padronização inicial 
# -----------------------
tb_orders_limpo = (
    bronze_tb_orders
    .select(
        F.col("order_id").cast("string").alias("id_pedido"),
        F.col("customer_id").cast("string").alias("id_consumidor"),
        F.col("order_status").cast("string").alias("status"),
        F.to_timestamp("order_purchase_timestamp").alias("data_compra"),
        F.to_timestamp("order_approved_at").alias("data_aprovacao"),
        F.to_timestamp("order_delivered_carrier_date").alias("data_entrega_transportadora"),
        F.to_timestamp("order_delivered_customer_date").alias("data_entrega_real"),
        F.to_timestamp("order_estimated_delivery_date").alias("data_entrega_estimada"),
        F.to_timestamp("timestamp_ingestion").alias("data_ingestao")
    )
    .withColumn("tempo_entrega_dias", F.datediff(F.col("data_entrega_real"), F.col("data_compra")))
    .withColumn("tempo_entrega_estimado_dias", F.datediff(F.col("data_entrega_estimada"), F.col("data_compra")))
    .withColumn("diferenca_entrega_dias", F.datediff(F.col("data_entrega_real"), F.col("data_entrega_estimada")))
    .withColumn("entrega_no_prazo", 
                F.when(F.col("diferenca_entrega_dias").isNull(), "Não Entregue")
                .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
                .otherwise("Não")
    )
)

# A fim de concluir a etapa de limpeza e padronização inicial, foi feito o replace dos valores do campo order_status pra português, a fim de manter a padronização atual dos dados
# 1. Cria um dicionário simples do Python com as traduções
dicionario_status = {
    "delivered": "entregue",
    "canceled": "cancelado",
    "shipped": "enviado",
    "processing": "processando",
    "invoiced": "faturado",
    "unavailable": "indisponível",
    "created": "criado",
    "approved": "aprovado"
}

# 2. Aplica a substituição na coluna alvo
tb_orders_traduzido = tb_orders_limpo.replace(to_replace=dicionario_status, subset=["status"])

display(tb_orders_limpo.limit(10))

print(f"Total de linhas na silver (tb_orders): {tb_orders_limpo.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
# Pra essa etapa, foram adicionadas 4 colunas :

# Tempo_entrega_dias = Diferença em dias do tempo de entrega real e o tempo de compra

# Tempo_entrega_estimado_dias = Diferença em dias do tempo de entrega estimada e o tempo de compra

# diferenca_entrega_dias = Diferença em dias do tempo de entrega real e o tempo de entrega estimada
    # Foi escolhido nessa ordem as datas devido a facilidade de entendimento por parte de outras pessoas que essa abordagem pode oferecer 

    # Cenário A (Entregou Atrasado): Entregou dia 12, mas era estimado para dia 10. Conta: 12 - 10 = 2. O resultado é positivo (+2).

    # Cenário B (Entregou Adiantado): Entregou dia 08, mas era estimado para dia 10. Conta: 8 - 10 = -2. O resultado é negativo (-2).

# entrega_no_prazo = Campo textual pra definir se foi ou não entrege no prazo 

tb_orders_validacao = (
    tb_orders_traduzido
    .withColumn("flag_id_pedido_valido", F.col("id_pedido").isNotNull())
    .withColumn("flag_id_consumidor_valido", F.col("id_consumidor").isNotNull())
    .withColumn("flag_status_valido", F.col("status").isin("entregue", "cancelado", "enviado", "processando", "faturado", "indisponível", "criado", "aprovado"))
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_id_pedido_valido") &
            F.col("flag_id_consumidor_valido") &
            F.col("flag_status_valido"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

display(tb_orders_validacao.limit(10))

# -----------------------
# Separação dos dados em validos e invalidos
# -----------------------
tb_orders_validos = tb_orders_validacao.filter(F.col("flag_qualidade") == "OK")
tb_orders_invalidos = tb_orders_validacao.filter(F.col("flag_qualidade") == "ERRO")

# -----------------------
# Escrita na camada silver
# -----------------------

# Dados válidos
tb_orders_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_pedidos_validos")

# Dados inválidos (mantidos para auditoria e rastreabilidade)
tb_orders_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_pedidos_invalidos")

print(f"✅ Tabela silver.fat_pedidos_validos criada com sucesso! ({tb_orders_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.fat_pedidos_invalidos criada para auditoria ({tb_orders_invalidos.count()} registros inválidos)")

## 3️⃣ silver.fat_itens_pedidos

In [0]:
bronze_tb_order_items = devolve_tabela_spark("tb_order_items")
bronze_tb_order_items.display(10)
bronze_tb_order_items.printSchema()

nulos_tb_order_items = bronze_tb_order_items.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_order_items.columns
]).display(10)

In [0]:
# -----------------------
# Limpeza e padronização inicial 
# -----------------------
tb_order_items_limpo = (
    bronze_tb_order_items
    .select(
        F.col("order_id").cast("string").alias("id_pedido"),
        F.col("order_item_id").cast("integer").alias("id_item_pedido"),
        F.col("product_id").cast("string").alias("id_produto"),
        F.col("seller_id").cast("string").alias("id_vendedor"),
        F.to_timestamp("shipping_limit_date").alias("data_entrega_limite"),
        F.col("price").cast("double").alias("preco"), 
        F.col("freight_value").cast("double").alias("frete"),
        F.to_timestamp("timestamp_ingestion").alias("data_ingestao")
    )
)

display(tb_order_items_limpo.limit(10))

print(f"Total de linhas na silver (tb_order_items): {tb_order_items_limpo.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
tb_order_items_validacao = (
    tb_order_items_limpo
    .withColumn("flag_id_pedido_valido", F.col("id_pedido").isNotNull())
    .withColumn("flag_id_pedido_item_valido", F.col("id_item_pedido").isNotNull())
    .withColumn("flag_produto_valido", F.col("id_produto").isNotNull())
    .withColumn("flag_vendedor_valido", F.col("id_vendedor").isNotNull())
    .withColumn("flag_preco_valido", F.col("preco") >= 0)
    .withColumn("flag_frete_valido", F.col("frete") >= 0)
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_id_pedido_valido") &
            F.col("flag_id_pedido_item_valido") &
            F.col("flag_produto_valido") &
            F.col("flag_vendedor_valido") &
            F.col("flag_preco_valido") &
            F.col("flag_frete_valido"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

display(tb_order_items_validacao.limit(10))

# -----------------------
# Separação dos dados em validos e invalidos
# -----------------------
tb_order_items_validos = tb_order_items_validacao.filter(F.col("flag_qualidade") == "OK")
tb_order_items_invalidos = tb_order_items_validacao.filter(F.col("flag_qualidade") == "ERRO")

# -----------------------
# Escrita na camada silver
# -----------------------

# Dados válidos
tb_order_items_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_itens_pedidos_validos")

# Dados inválidos (mantidos para auditoria e rastreabilidade)
tb_order_items_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_itens_pedidos_invalidos")

print(f"✅ Tabela silver.fat_itens_pedidos_validos criada com sucesso! ({tb_order_items_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.fat_itens_pedidos_invalidos criada para auditoria ({tb_order_items_invalidos.count()} registros inválidos)")

## 4️⃣ silver.fat_pagamentos_pedidos

In [0]:
bronze_tb_order_payments = devolve_tabela_spark("tb_order_payments")
bronze_tb_order_payments.display(10)
bronze_tb_order_payments.printSchema()

nulos_tb_order_payments = bronze_tb_order_payments.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_order_payments.columns
]).display(10)

# Pegando quais os tipos de pagaento utilizados 
df  = bronze_tb_order_payments
display(df.select("payment_type").distinct())

In [0]:
# -----------------------
# Limpeza e padronização inicial 
# -----------------------
tb_order_payments_limpo = (
    bronze_tb_order_payments
    .select(
        F.col("order_id").cast("string").alias("id_pedido"),
        F.col("payment_sequential").cast("integer").alias("sequencial_pagamento"),
        F.col("payment_type").cast("string").alias("tipo_pagamento"),
        F.col("payment_installments").cast("integer").alias("parcelas"),
        F.col("payment_value").cast("double").alias("valor_pagamento"),
        F.to_timestamp("timestamp_ingestion").alias("data_ingestao")
    )
)

# A fim de concluir a etapa de limpeza e padronização inicial, foi feito o replace dos valores do campo order_status pra português, a fim de manter a padronização atual dos dados
# 1. Cria um dicionário simples do Python com as traduções
dicionario_pagamento = {
    "credit_card": "Cartão de Crédito",
    "boleto": "Boleto",
    "voucher": "Voucher",
    "debit_card": "Cartão de Débito",
    "not_defined": "Não Definido",
}

# 2. Aplica a substituição na coluna alvo
tb_order_payments_traduzido = tb_order_payments_limpo.replace(to_replace=dicionario_pagamento, subset=["tipo_pagamento"])

display(tb_order_payments_traduzido.limit(10))

print(f"Total de linhas na silver (tb_order_payments): {tb_order_payments_traduzido.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
tb_order_payments_validacao = (
    tb_order_payments_traduzido
    .withColumn("flag_id_pedido_valido", F.col("id_pedido").isNotNull())
    .withColumn("flag_tipo_pagamento_valido", F.col("tipo_pagamento").isin("Cartão de Crédito", "Boleto", "Voucher", "Cartão de Débito", "Não Definido"))
    .withColumn("flag_preco_valido", F.col("valor_pagamento") >= 0)
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_id_pedido_valido") &
            F.col("flag_tipo_pagamento_valido") &
            F.col("flag_preco_valido"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

display(tb_order_payments_validacao.limit(10))

# -----------------------
# Separação dos dados em validos e invalidos
# -----------------------
tb_order_payments_validos = tb_order_payments_validacao.filter(F.col("flag_qualidade") == "OK")
tb_order_payments_invalidos = tb_order_payments_validacao.filter(F.col("flag_qualidade") == "ERRO")

# -----------------------
# Escrita na camada silver
# -----------------------

# Dados válidos
tb_order_payments_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_pagamentos_pedidos_validos")

# Dados inválidos (mantidos para auditoria e rastreabilidade)
tb_order_payments_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_pagamentos_pedidos_invalidos")

print(f"✅ Tabela silver.fat_pagamentos_pedidos_validos criada com sucesso! ({tb_order_payments_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.fat_pagamentos_pedidos_invalidos criada para auditoria ({tb_order_payments_invalidos.count()} registros inválidos)")

## 5️⃣ silver.fat_avaliacoes_pedidos

In [0]:
bronze_tb_order_reviews = devolve_tabela_spark("tb_order_reviews")
bronze_tb_order_reviews.display(10)
bronze_tb_order_reviews.printSchema()

nulos_tb_order_reviews = bronze_tb_order_reviews.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_order_reviews.columns
])

nulos_tb_order_reviews.display(10)
print(f"Quantidade de registros na camada bronze: {bronze_tb_order_reviews.count()}")

In [0]:
# -----------------------
# Limpeza e padronização inicial 
# -----------------------
tb_order_reviews_limpo = (
    bronze_tb_order_reviews
    .select(
        F.col("review_id").cast("string").alias("id_avaliacao"),
        F.col("order_id").cast("string").alias("id_pedido"),
        F.col("review_score").cast("integer").alias("nota_avaliacao"),
        F.coalesce(F.col("review_comment_title").cast("string"), F.lit("Sem título")).alias("titulo_comentario_avaliacao"),
        F.coalesce(F.col("review_comment_message").cast("string"), F.lit("Sem comentário")).alias("mensagem_comentario_avaliacao"),
        F.try_to_timestamp("review_creation_date").alias("data_criacao_comentario"),
        F.try_to_timestamp("review_answer_timestamp").alias("data_resposta_comentario"),
        F.to_timestamp("timestamp_ingestion").alias("data_ingestao")
    )
)

display(tb_order_reviews_limpo.limit(10))

print(f"Total de linhas na silver (tb_order_reviews): {tb_order_reviews_limpo.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
tb_order_reviews_validacao = (
    tb_order_reviews_limpo
    .withColumn("flag_id_avaliacao_valido", F.col("id_avaliacao").isNotNull())
    .withColumn("flag_id_pedido_valido", F.col("id_pedido").isNotNull())
    .withColumn("flag_nota_avaliacao_valido", (F.col("nota_avaliacao") >= 0) & (F.col("nota_avaliacao") <= 5))
    .withColumn("flag_data_criacao_valida", F.col("data_criacao_comentario") <= F.current_timestamp())
    .withColumn("flag_data_resposta_valida", F.col("data_resposta_comentario") <= F.current_timestamp())
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_id_avaliacao_valido") &
            F.col("flag_id_pedido_valido") &
            F.col("flag_nota_avaliacao_valido") &
            F.col("flag_data_criacao_valida") &
            F.col("flag_data_resposta_valida"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

display(tb_order_reviews_validacao.limit(10))

# -----------------------
# Separação dos dados em validos e invalidos
# -----------------------
tb_order_reviews_validos = tb_order_reviews_validacao.filter(F.col("flag_qualidade") == "OK")
tb_order_reviews_invalidos = tb_order_reviews_validacao.filter(F.col("flag_qualidade") == "ERRO")

# -----------------------
# Escrita na camada silver
# -----------------------

# Dados válidos
tb_order_reviews_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_avaliacoes_pedidos_validos")

# Dados inválidos (mantidos para auditoria e rastreabilidade)
tb_order_reviews_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_avaliacoes_pedidos_invalidos")

print(f"✅ Tabela silver.fat_avaliacoes_pedidos_validos criada com sucesso! ({tb_order_reviews_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.fat_avaliacoes_pedidos_invalidos criada para auditoria ({tb_order_reviews_invalidos.count()} registros inválidos)")

## 6️⃣ silver.dim_produtos

In [0]:
bronze_tb_products = devolve_tabela_spark("tb_products")
bronze_tb_products.display(10)
bronze_tb_products.printSchema()

nulos_tb_products = bronze_tb_products.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_products.columns
]).display(10)

In [0]:
window_spec = Window.partitionBy("id_produto").orderBy(F.col("data_ingestao").desc())

# -----------------------
# Limpeza e padronização inicial 
# -----------------------
tb_products_limpo = (
    bronze_tb_products
    .select(
        F.col("product_id").cast("string").alias("id_produto"),
        F.initcap(F.trim(F.col("product_name"))).cast("string").alias("nome_produto"),
        
        # Coluna de Texto: Usamos "Não Informado"
        F.upper(F.coalesce(F.col("product_category_name").cast("string"), F.lit("Não Informado"))).alias("categoria_produto"),
    
        # Mantemos null pois poderia haver impacto negativo de imputar qualquer valor e além disso não sabemos qual é o cálculo realizado para sabero tamanho do nome do produto (não há correlação direta com o nome na tabela)
        F.col("product_name_lenght").cast("integer").alias("tamanho_nome_produto"),
        F.coalesce(F.col("product_description_lenght").cast("integer"), F.lit(0)).alias("tamanho_descricao_produto"),

        # Fotos podem ser 0 (realidade de negócio)
        F.coalesce(F.col("product_photos_qty").cast("integer"), F.lit(0)).alias("quantidade_fotos"),
        
        # Mantemos NULL para dimensões físicas (um produto não pode ter 0 gramas ou 0 cm)
        F.col("product_weight_g").cast("integer").alias("peso_produto_gramas"),
        F.col("product_length_cm").cast("integer").alias("comprimento_centimetros"),
        F.col("product_height_cm").cast("integer").alias("altura_centimetros"),
        F.col("product_width_cm").cast("integer").alias("largura_centimetros"),
        
        F.to_timestamp("timestamp_ingestion").alias("data_ingestao")
    )
    # 1. Aplicando a window function e armazenando em coluna temporária 
    .withColumn("rn", F.row_number().over(window_spec))
    
    # 2. Filtramos mantendo APENAS a linha número 1 (a mais recente de cada ID)
    .filter(F.col("rn") == 1)
    
    # 3. A coluna 'rn' foi removida pois ela era apenas um apoio para nos ajudar no filtro
    .drop("rn")
)

display(tb_products_limpo.limit(10))

print(f"Total de linhas na silver (tb_products): {tb_products_limpo.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
tb_products_validacao = (
    tb_products_limpo
    .withColumn("flag_id_valido", F.col("id_produto").isNotNull())
    .withColumn("flag_nome_valido", F.col("nome_produto").isNotNull() & (F.length(F.col("nome_produto")) > 1))
    .withColumn("flag_qtd_fotos_valida", F.col("quantidade_fotos") >= 0)
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_id_valido") &
            F.col("flag_nome_valido") &
            F.col("flag_qtd_fotos_valida"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

display(tb_products_validacao.limit(10))

# -----------------------
# Separação dos dados em validos e invalidos
# -----------------------
tb_products_validos = tb_products_validacao.filter(F.col("flag_qualidade") == "OK")
tb_products_invalidos = tb_products_validacao.filter(F.col("flag_qualidade") == "ERRO")

# -----------------------
# Escrita na camada silver
# -----------------------

# Dados válidos
tb_products_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_produtos_validos")

# Dados inválidos (mantidos para auditoria e rastreabilidade)
tb_products_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_produtos_invalidos")

print(f"✅ Tabela silver.dim_produtos_validos criada com sucesso! ({tb_products_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.dim_produtos_invalidos criada para auditoria ({tb_products_invalidos.count()} registros inválidos)")

## 7️⃣ silver.dim_vendedores

In [0]:
bronze_tb_sellers = devolve_tabela_spark("tb_sellers")
bronze_tb_sellers.display(10)
bronze_tb_sellers.printSchema()

nulos_tb_sellers = bronze_tb_sellers.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_sellers.columns
]).display(10)

In [0]:
# Window Function que utilizada para filtrar a bronze procurando por duplicatas na coluna id_consumidor usando como referência pra odernação a coluna de data_ingestao 
window_spec = Window.partitionBy("id_vendedor").orderBy(F.col("data_ingestao").desc())

# -----------------------
# Limpeza e padronização inicial 
# -----------------------
tb_sellers_limpo = (
    bronze_tb_sellers
    .select(
        F.col("seller_id").cast("string").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").cast("integer").alias("prefixo_cep"),
        F.upper(F.col("seller_city").cast("string")).alias("cidade"),
        F.upper(F.col("seller_state").cast("string")).alias("estado"),
        F.initcap(F.trim(F.col("seller_name"))).cast("string").alias("nome_vendedor"),
        F.to_date("seller_registration_date").alias("data_registro_vendedor"),
        F.to_timestamp("timestamp_ingestion").alias("data_ingestao")
    )
    # 1. Aplicando a window function e armazenando em coluna temporária 
    .withColumn("rn", F.row_number().over(window_spec))
    
    # 2. Filtramos mantendo APENAS a linha número 1 (a mais recente de cada ID)
    .filter(F.col("rn") == 1)
    
    # 3. A coluna 'rn' foi removida pois ela era apenas um apoio para nos ajudar no filtro
    .drop("rn")
)

display(tb_sellers_limpo.limit(10))

print(f"Total de linhas na silver (tb_sellers_limpo): {tb_sellers_limpo.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
tb_sellers_validacao = (
    tb_sellers_limpo
    .withColumn("flag_id_vendedor", F.col("id_vendedor").isNotNull())
    .withColumn("flag_cep_valido", F.col("prefixo_cep") > 0)
    .withColumn("flag_nome_vendedor_valido", F.col("nome_vendedor").isNotNull() & (F.length(F.col("nome_vendedor")) > 1))
    .withColumn("flag_reistration_date_valido", F.col("data_registro_vendedor") <= F.current_date())
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_id_vendedor") &
            F.col("flag_cep_valido") &
            F.col("flag_nome_vendedor_valido") &
            F.col("flag_reistration_date_valido"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

display(tb_sellers_validacao.limit(10))

# -----------------------
# Separação dos dados em validos e invalidos
# -----------------------
tb_sellers_validos = tb_sellers_validacao.filter(F.col("flag_qualidade") == "OK")
tb_sellers_invalidos = tb_sellers_validacao.filter(F.col("flag_qualidade") == "ERRO")

# -----------------------
# Escrita na camada silver
# -----------------------

# Dados válidos
tb_sellers_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_vendedores_validos")

# Dados inválidos (mantidos para auditoria e rastreabilidade)
tb_sellers_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_vendedores_invalidos")

print(f"✅ Tabela silver.dim_vendedores_validos criada com sucesso! ({tb_sellers_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.dim_vendedores_invalidos criada para auditoria ({tb_sellers_invalidos.count()} registros inválidos)")

## 8️⃣ silver.dim_categoria_produtos_traducao

In [0]:
bronze_tb_product_category_name_translation = devolve_tabela_spark("tb_product_category_name_translation")
bronze_tb_product_category_name_translation.display(10)
bronze_tb_product_category_name_translation.printSchema()

nulos_tb_product_category_name_translation = bronze_tb_product_category_name_translation.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_product_category_name_translation.columns
]).display(10)

In [0]:
# -----------------------
# Limpeza e padronização inicial 
# -----------------------
tb_product_category_name_translation_limpo = (
    bronze_tb_product_category_name_translation
    .select(
        F.upper(F.col("product_category_name").cast("string")).alias("nome_produto_pt"),
        F.upper(F.col("product_category_name_english").cast("string")).alias("nome_produto_en"),
        F.to_timestamp("timestamp_ingestion").alias("data_ingestao")
    )
)

print(f"Total de linhas na silver (tb_product_category_name_translation): {tb_product_category_name_translation_limpo.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
tb_product_category_name_translation_validacao = (
    tb_product_category_name_translation_limpo
    .withColumn("flag_nome_produto_pt_valido", F.col("nome_produto_pt").isNotNull())
    .withColumn("flag_nome_produto_en_valido", F.col("nome_produto_en").isNotNull())
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_nome_produto_pt_valido") &
            F.col("flag_nome_produto_en_valido"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

display(tb_product_category_name_translation_validacao.limit(10))

# -----------------------
# Separação dos dados em validos e invalidos
# -----------------------
tb_product_category_name_translation_validos = tb_product_category_name_translation_validacao.filter(F.col("flag_qualidade") == "OK")
tb_product_category_name_translation_invalidos = tb_product_category_name_translation_validacao.filter(F.col("flag_qualidade") == "ERRO")

# -----------------------
# Escrita na camada silver
# -----------------------

# Dados válidos
tb_product_category_name_translation_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_categoria_produtos_traducao_validos")

# Dados inválidos (mantidos para auditoria e rastreabilidade)
tb_product_category_name_translation_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_categoria_produtos_traducao_invalidos")

print(f"✅ Tabela silver.dim_categoria_produtos_traducao_validos criada com sucesso! ({tb_product_category_name_translation_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.dim_categoria_produtos_traducao_invalidos criada para auditoria ({tb_product_category_name_translation_invalidos.count()} registros inválidos)")

## 9️⃣ silver.dim_cotacao_dolar

In [0]:
bronze_tb_cotacao_dolar = devolve_tabela_spark("tb_cotacao_dolar")
bronze_tb_cotacao_dolar.display(10)
bronze_tb_cotacao_dolar.printSchema()

nulos_tb_cotacao_dolar = bronze_tb_cotacao_dolar.select([
  F.count(F.when(F.col(c).isNull(), c)).alias(c)
  for c in bronze_tb_cotacao_dolar.columns
])

print(f"Qtd de dados na tabela bronze: {bronze_tb_cotacao_dolar.count()}")
nulos_tb_cotacao_dolar.display(10)


In [0]:
# ----------------------------------------------------------------------------------------------
# O primeiro passo é entender pra construir essa tabela na camada silver é entender que a API só retorna valores da cotação do dólar de Segunda a Sexta, não colocando nada pros finais de semana

# Portanto, para conseguirmos alcançar o objteivo precisamos preencher os buracos do fim de semana. E pra isso acontecer, é necessário que os finais de semana existam na tabela

# Para concretizar a primeira parte do nosso plano, usamos F.min() e F.max() para descobrir o primeiro e o último dia que vieram da API.

# Em seguida criamos um array contendo todos os dias entre o mínimo e o máximo (F.sequence) e usamos o F.explode para "explodir" esse array, transformando cada data em uma linha nova.

# O Resultado: Agora temos um calendário perfeito, sem faltar um único dia.

# 1. Obtendo df com as datas corretas 

bronze_referencia = bronze_tb_cotacao_dolar.withColumn(
    "data_referencia", 
    F.to_date("dataHoraCotacao") # Corta as horas fora
)

limites_data = bronze_referencia.agg(
    F.min("data_referencia").alias("inicio"),
    F.max("data_referencia").alias("fim")
).collect()[0]

# Gera um DataFrame com todas as datas contínuas entre o inicio e o fim
df_calendario = (
    spark.createDataFrame([(limites_data["inicio"], limites_data["fim"])], ["inicio", "fim"])
    .select(F.explode(F.sequence(F.to_date("inicio"), F.to_date("fim"))).alias("data_referencia"))
)

# ----------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------

# 2. Join e Preenchimento (Forward Fill)
# Após a etapa 1, fazemos o left join do calendário que possui todos os dias com a tabela da bronze que possuimos. Dessa forma conseguimos um dataframe com todos os dias da semana. Sábados e Domingos ficam NULL.
tb_dolar_com_nulos = df_calendario.join(
    bronze_referencia, 
    on="data_referencia", 
    how="left"
)

# ----------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------

# 3. Cria a Window Function que olha do começo até a linha atual

# Window.orderBy("data_cotacao"): Primeiro, mandamos o Spark organizar os dados cronologicamente. O tempo precisa fluir na direção certa.

# rowsBetween(Window.unboundedPreceding, Window.currentRow): Essa é a chave de ouro. Por padrão, algumas funções de janela olham apenas para a linha atual. Nós forçamos a janela a olhar desde o início dos tempos (unboundedPreceding) até a linha atual.

window_ffill = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)

tb_cotacao_limpa = (
    tb_dolar_com_nulos
    # Preenche a cotação de compra
    .withColumn("cotacaoCompra", F.last("cotacaoCompra", ignorenulls=True).over(window_ffill))
    
    # Restaura a coluna dataHoraCotacao original. 
    # Nos finais de semana (onde é nulo), inventamos a hora "23:59:59" apenas para manter o padrão
    .withColumn(
        "dataHoraCotacao", 
        F.coalesce(F.col("dataHoraCotacao"), F.to_timestamp(F.concat(F.col("data_referencia"), F.lit(" 23:59:59"))))
    )
    
    # Garante o timestamp de ingestão válido
    .withColumn(
        "data_ingestao", 
        F.coalesce(F.col("timestamp_ingestion"), F.current_timestamp())
    )
    
    # Limpa as colunas que não precisamos mais (a original de ingestão e a referência do join)
    .drop("timestamp_ingestion", "data_referencia")
)

display(tb_cotacao_limpa.limit(10))

print(f"Total de linhas na silver (tb_cotacao): {tb_cotacao_limpa.count()}")

# -----------------------
# Regras de qualidade
# -----------------------
tb_cotacao_validacao = (
    tb_cotacao_limpa
    .withColumn("flag_data_valida", F.col("dataHoraCotacao").isNotNull())
    .withColumn("flag_cotacao_valida", F.col("cotacaoCompra").isNotNull() & (F.col("cotacaoCompra") > 0))
    .withColumn("flag_qualidade", 
        F.when(
            F.col("flag_data_valida") &
            F.col("flag_cotacao_valida"),
            F.lit("OK")
        ).otherwise(F.lit("ERRO"))
    )
)

# -----------------------
# Separação e Escrita na camada Silver
# -----------------------
tb_cotacao_validos = tb_cotacao_validacao.filter(F.col("flag_qualidade") == "OK")
tb_cotacao_invalidos = tb_cotacao_validacao.filter(F.col("flag_qualidade") == "ERRO")

tb_cotacao_validos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_cotacao_dolar_validos")
tb_cotacao_invalidos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.dim_cotacao_dolar_invalidos")

print(f"✅ Tabela silver.dim_cotacao_dolar_validos criada com sucesso! ({tb_cotacao_validos.count()} registros válidos)")
print(f"⚠️ Tabela silver.dim_cotacao_dolar_invalidos criada para auditoria ({tb_cotacao_invalidos.count()} registros inválidos)")

## 🔟 silver.fat_pedido_total

In [0]:
# 1. Leitura das Tabelas Base para silver final 
tb_orders = spark.table(f"{catalogo}.{schema_silver}.fat_pedidos_validos")
tb_payments = spark.table(f"{catalogo}.{schema_silver}.fat_pagamentos_pedidos_validos")
tb_cotacao = spark.table(f"{catalogo}.{schema_silver}.dim_cotacao_dolar_validos")

# 2. Agrupamento dos Pagamentos
# Nessa etapa criamos um dataframe que possua o id do pedido e o valor total pago em reais agrupado por pedido 
df_pagamentos = tb_payments.groupBy("id_pedido").agg(
    F.sum("valor_pagamento").alias("valor_total_pago_brl")
)

# 2.1. Verificação rápida do df 
df_pagamentos.display(10)

# 3. Construção da Tabela Final
df_fat_pedido_total = (
    tb_orders
    
    # A. Join com Pagamentos 
    .join(df_pagamentos, on="id_pedido", how="left")
    
    # B. Se o pedido não teve pagamento, o valor fica 0.0 (evita nulos na matemática)
    # Além disso, dessa forma ele já ficaria com 2 casas decimais, que é o que queremos
    .withColumn("valor_total_pago_brl", F.coalesce(F.col("valor_total_pago_brl"), F.lit(0.0)))
    
    # C. Criar uma coluna de data para conseguir cruzar com a cotação
    # A coluna usada pra isso foi a coluna de data_compra advinda da tabela de pedidos
    .withColumn("data_join", F.to_date(F.col("data_compra"))) 
    
    # D. Join com a Cotação do Dólar 
    .join(
        tb_cotacao, 
        F.col("data_join") == F.to_date(F.col("dataHoraCotacao")), 
        how="left"
    )
    
    # E. Calculando o valor em USD (BRL dividido pela cotação do dia)
    .withColumn(
        "valor_total_pago_usd", 
        F.round(F.col("valor_total_pago_brl") / F.col("cotacaoCompra"), 2) # O round e o 2 servem pra definir quantas casas decimais serão utilizadas após a vírgula/ponto
    )
    
    # Selecionando e renomeando (se necessário) as colunas
    .select(
        "id_pedido",
        "id_consumidor",
        "status",
        "valor_total_pago_brl",
        "valor_total_pago_usd",
        F.to_timestamp("data_compra").alias("data_pedido")
    )
)

# Display no result final para validação
df_fat_pedido_total.display(10)

# 4. Ingerindo a tabela na camada Silver
df_fat_pedido_total.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{schema_silver}.fat_pedido_total")

## Otimização Física com Delta Optimize 👍
- No final do notebook da Camada Silver, execute o
comando de manutenção do Delta Lake (OPTIMIZE com ZORDER) nas principais tabelas fato,
indexando pelas colunas de id_pedido e data_pedido para garantir alta performance
analítica.

In [0]:
%sql

OPTIMIZE medalhao_at.silver.fat_pedidos_validos
ZORDER BY (id_pedido, data_compra);

OPTIMIZE medalhao_at.silver.fat_itens_pedidos_validos
ZORDER BY (id_pedido);

OPTIMIZE medalhao_at.silver.fat_pagamentos_pedidos_validos
ZORDER BY (id_pedido);

OPTIMIZE medalhao_at.silver.fat_avaliacoes_pedidos_validos
ZORDER BY (id_avaliacao, data_criacao_comentario);

OPTIMIZE medalhao_at.silver.fat_pedido_total
ZORDER BY (id_pedido, data_pedido);

## Etapa especial: Remoção das tabelas inválidas 
- Após analisar cada uma da ingestões realizadas na camada silver, percebeu-se que nenhuma das tabelas da camada silver dedicadas a auditoria possui algum tipo de dado. Portanto, a fim de evitar uso desnecessário de armazenamento, elas serão deletadas
- **OBS: CASO O COTROLADOR DO PIPELINE JULGAR ÚTIL MANTER AS TABELAS, BASTA IGNORAR A CÉLULA DE CÓDIGO ABAIXO**

In [0]:
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.dim_categoria_produtos_traducao_invalidos")
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.dim_consumidores_invalidos")
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.dim_produtos_invalidos")
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.dim_vendedores_invalidos")
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.fat_avaliacoes_pedidos_invalidos")
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.fat_itens_pedidos_invalidos")
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.fat_pagamentos_pedidos_invalidos")
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.dim_cotacao_dolar_invalidos")
spark.sql("DROP TABLE IF EXISTS medalhao_at.silver.fat_pedidos_invalidos")